# Asynchronous model fitting: Calculating likelihood

Fitting models with an asynchronous assumption (i.e., that only one spin can update at a time) requires a different approach to the synchronous assumption (all spins update simultaneously). If multiple spins can update in a single timestep, we must account for all possible ways that this may happen.

Let us assume that a spin can update _at most_ once per timestep. Then given a pair of consecutive states $\boldsymbol{s}^t, \boldsymbol{s}^{t+1}$, the conditonal likelihood $P(\boldsymbol{s}^{t+1} \mid \boldsymbol{s}^t)$ is the summation of probabilities for all of the ways that we may transition from $\boldsymbol{s}^t$ to $\boldsymbol{s}^{t+1}$.

If we know that a spin has updated (i.e., that $s_i^t \neq s_i^{t+1}$), this gives us some information. We know that this spin certainly updated, so the only question that remains is: "when?".

For all other spins, we cannot assume that they did or did not update. We have to consider all options. 

In [ ]:
from itertools import combinations, permutations

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt

from ising import Ising

type Configuration = npt.NDArray[np.int64]

RANDOM_SEED = 202606081437

Let's first consider a simple example:

$$\boldsymbol{s}^t = \begin{pmatrix}-1 & 1 & 1\end{pmatrix} \qquad \boldsymbol{s}^{t+1} = \begin{pmatrix}1 & 1 & 1\end{pmatrix}$$

We know that $s_1$ certainly updated. We must consider the cases where the others spins did or did not update. For each possible combination, we must consider all possible orderings.

In [ ]:
s0 = np.array([-1, 1, 1])
s1 = np.array([1, 1, 1])

1. We first figure out which spins certainly updated, and which ones only maybe updated.

In [ ]:
def transition_spins(
    s0: Configuration, s1: Configuration
) -> tuple[npt.NDArray[np.int64], npt.NDArray[np.int64]]:
    nonequal = s0 != s1
    return np.argwhere(nonequal).flatten(), np.argwhere(~nonequal).flatten()

In [ ]:
did_update, maybe_update = transition_spins(s0, s1)

print(f"Certainly updated: {did_update}")
print(f"Maybe updated: {maybe_update}")

2. Now we determine all combinations of spins from the 'maybe' pile which could have been chosen to update. We can do this easily with the `itertools.combinations` function.

In [ ]:
possible_update_sets = [
    combo
    for r in range(len(maybe_update) + 1)
    for combo in combinations([int(s) for s in maybe_update], r)
]
possible_update_sets

In [ ]:
possible_update_sets = [
    combo for combo in combinations([int(s) for s in maybe_update], len(maybe_update))
]
possible_update_sets

Which we can combine with our set of spins that we know definitely updated, to get our collection of sets of possible update spins.

In [ ]:
possible_update_sets = [
    sorted(tuple(int(x) for x in did_update) + combo) for combo in possible_update_sets
]
possible_update_sets

3. Thirdly, for each of these sets, we have to consider all orders in which the spins might have updated. This time we'll use `itertools.permutations`.

In [ ]:
possible_orderings = [
    order for update_set in possible_update_sets for order in permutations(update_set)
]
print(possible_orderings)

## Calculating the likelihood

Fortunately, we can rely on some existing machinery in the `Ising` class. For simplicity we will fix the thresholds $\boldsymbol{h}$ and interactions $\boldsymbol{J}$ here, but in the real implementation they would come from the optimiser.

In [ ]:
h = np.array([-0.3, 0.4, 0.7])
J = np.array(
    [
        [1.0, 0.5, 0.5],
        [0.5, 1.0, 0.0],
        [0.0, 0.5, 1.0],
    ]
)
adj = np.array(
    [
        [True, True, True],
        [True, True, False],
        [False, True, True],
    ]
)

# X is a covariate vector, but we won't use that here. We'll just use the intercept,
#   so set X to a vector of 1s.
X = np.ones(1, dtype=np.float64)

Remember that we've assumed each spin updates at most once per measurement interval. Once we've fixed an update set, we're assuming that _all_ of those spins have updated. This means that for spins which we know updated, we're looking at the probability that they _did_ update. For the remaining spins, we're calculating the probability that they _did not_ update.

In any case, it boils down to running the updates in order, transitioning the spins from $\boldsymbol{s}^t$ to those in $\boldsymbol{s}^{t+1}$, calculating the probability of transition at each step, and multiplying these together.

In [ ]:
def calculate_ordering_probability(
    order: tuple[int],
    # updated: npt.NDArray[np.bool],
    s0: Configuration,
    s1: Configuration,
    h: npt.NDArray[np.float64],
    J: npt.NDArray[np.float64],
    adj: npt.NDArray[np.bool],
    X: npt.NDArray[np.float64],
) -> float:
    if len(order) == 0:
        return 1.0

    log_prob = 0.0
    prev = s0.copy()
    for spin_idx in order:
        theta = Ising.glauber_theta(prev, X, i=spin_idx, h=h, j=J, adj=adj)
        new_state = prev
        new_state[spin_idx] = s1[spin_idx]
        log_prob += new_state[spin_idx] * theta - np.log(2 * np.cosh(theta))

    return log_prob


def log_likelihood(
    orderings: list[tuple[int, ...]],
    s0: Configuration,
    s1: Configuration,
    h: npt.NDArray[np.float64],
    J: npt.NDArray[np.float64],
    adj: npt.NDArray[np.bool],
    X: npt.NDArray[np.float64],
) -> float:
    p = 0.0
    for order in orderings:
        ordering_log_p = calculate_ordering_probability(
            order,
            s0,
            s1,
            h,
            J,
            adj,
            X,
        )
        ordering_p = np.exp(np.log(1 / len(orderings)) + ordering_log_p)
        p += ordering_p
    return np.log(p)

In [ ]:
log_likelihood(possible_orderings, s0, s1, h, J, adj, X)

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
repeats = 100
samples = np.zeros(repeats, dtype=np.float64)
for r in range(repeats):
    perms = rng.choice(possible_orderings, size=3, replace=False)
    samples[r] = log_likelihood(perms, s0, s1, h, J, adj, X)


fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
ax.hist(samples)
ax.axvline(log_likelihood(possible_orderings, s0, s1, h, J, adj, X))

In [ ]:
import itertools
import time

import numpy as np


def benchmark_python(iterations: int = 100):
    rng = np.random.default_rng(42)

    m = 1600
    t = 2
    n = 6
    p = 1

    # Match Rust benchmark data generation
    y = rng.choice(
        np.array([-1, 1], dtype=np.int64),
        size=(m, t, n),
    )

    X = rng.random((m, t, p), dtype=np.float64)

    h = rng.random(n, dtype=np.float64)

    j = rng.random((n, n), dtype=np.float64)

    adj = rng.choice(
        np.array([False, True]),
        size=(n, n),
    )

    # All permutations of 0..n-1
    possible_orderings = np.array(
        list(itertools.permutations(range(n))),
        dtype=np.int64,
    )

    # Warm-up JIT compilation
    result = Ising.time_series_nll_async(
        y,
        X,
        h,
        j,
        adj,
        possible_orderings[:64],
    )

    start = time.perf_counter()

    for _ in range(iterations):
        result = Ising.time_series_nll_async(
            y,
            X,
            h,
            j,
            adj,
            possible_orderings[:64],
        )

    elapsed = time.perf_counter() - start

    print(f"result = {result}")
    print(f"iterations = {iterations}")
    print(f"total = {elapsed:.6f}s")
    print(f"per iteration = {elapsed / iterations:.6f}s")


benchmark_python()